# ==============================================
# ASSIGNMENT 2: An Investigation of the Components of Pre-Training and Post-Training in LLMs
# ==============================================

# Instructions:
## 1. Use Google Colab for all experiments (free GPU tier is sufficient).
## 2. This notebook provides a complementory solution for all parts of the assignment.

# ==============================================
# SUBMISSION INSTRUCTION
# ==============================================

## 1. Please write the name of the file as `Group_(number)_assignemnt_2_solution.ipynb`

##2. Only one member from one group needs to submit the solution, to avoid any duplicasy.

## **Question 1: Analyzing with or without Adapter fine-tuning for Multi-Document Summarization (MDS) task.**

## PART 1, 2 and 3

In [ ]:
# Force a clean environment by removing old cache and reinstalling a stable version
!rm -rf /root/.cache/huggingface/datasets
!pip install -q datasets==2.18.0

# Set a safe cache location to avoid file system errors
import os
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache"

In [ ]:
!pip install -q rouge_score

In [ ]:
import nltk
nltk.download('punkt_tab')

In [ ]:
%env CUDA_LAUNCH_BLOCKING=1

In [ ]:
# ============================================================
# Imports and Initialization
# ============================================================

import os, math, json, random
from typing import List
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer, BartForConditionalGeneration
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# -------------------------
# Config
# -------------------------
BACKBONE = "facebook/bart-base"
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 128
DOC_SEP = "</s>"

ADAPTER_TYPE = "residual"       # choose: "residual" or "sparse"
N_ADAPTER_LAYERS = 3            # top-N layers to adapt (adapters)
ADAPTER_BOTTLENECK = 128
SPARSE_TOPK = 128
USE_FRACTION_FOR_SPARSE = False
SPARSITY_LAMBDA = 1e-5          # L1 regularization

TRAIN_SUBSET = 2000
VAL_SUBSET = 500
BATCH_SIZE = 32                    # or 16 or 8
EPOCHS = 5                         # or 8
LR = 3e-5
CHECKPOINT_DIR = "/content/bart_adapters_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

scorer_eval = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

In [ ]:
# ============================================================
# Utilities
# ============================================================
def flatten_abstracts_to_text(background: str, abstracts: List[str]):
    if background and background.strip():
        return background.strip() + " " + DOC_SEP + " " + (" " + DOC_SEP + " ").join(abstracts)
    return (" " + DOC_SEP + " ").join(abstracts)

def tokenize_texts(tokenizer, texts, max_length):
    return tokenizer(texts, truncation=True, padding="longest",
                     max_length=max_length, return_tensors="pt")

In [ ]:
# ============================================================
# Adapter Modules
# ============================================================
class ResidualAdapter(nn.Module):
    def __init__(self, hidden_size, bottleneck=64):
        super().__init__()
        # TODO: Define down-projection (hidden_size → bottleneck)
        # TODO: Define up-projection (bottleneck → hidden_size)
        # TODO: Initialize up layer weights and biases to zero
    def forward(self, x):
        # TODO: Apply down-projection and ReLU activation
        # TODO: Apply up-projection to get residual delta
        return x + delta

class SparseAdapter(nn.Module):
    def __init__(self, hidden_size, bottleneck=64, topk=128, use_fraction=False):
        super().__init__()
        # TODO: Define down-projection (hidden_size → bottleneck)
        # TODO: Define up-projection (bottleneck → hidden_size)
        # TODO: Initialize up layer weights and biases to zero
        # TODO: Define gating network for computing token importance scores
        # TODO: Store topk and use_fraction parameters

    def sample_gumbel(self, shape, device='cpu', eps=1e-20):
        U = torch.rand(shape, device=device)
        return -torch.log(-torch.log(U + eps) + eps)

    def relaxed_topk_mask(self, logits: torch.Tensor, k: int, tau: float = 1.0):
        if logits.dim() == 1:
            logits = logits.unsqueeze(0)
        device = logits.device
        g = self.sample_gumbel(logits.shape, device=device)
        y = (logits + g) / tau
        probs = F.softmax(y, dim=-1)
        topk_idx = torch.topk(logits, k=k, dim=-1).indices
        hard_mask = torch.zeros_like(logits)
        hard_mask[0, topk_idx[0]] = 1.0
        mask = (hard_mask - probs).detach() + probs
        return mask.squeeze(0)

    def forward(self, x):
        # TODO: Compute gating logits from input
        # TODO: Build relaxed top-k masks for each batch element
        # TODO: Apply down-projection, activation, and up-projection
        # TODO: Multiply by mask and add residual correction to input
        # TODO: Return adapted output

    def l1_penalty(self):
        return torch.norm(self.down.weight, 1) + torch.norm(self.up.weight, 1)

# ============================================================
# Adapter Injection Helpers
# ============================================================
import types

def wrap_encoder_layer(layer, adapter):
    original_forward = layer.forward
    def forward_with_adapter(self, hidden_states, *args, **kwargs):
        outputs = original_forward(hidden_states, *args, **kwargs)
        if isinstance(outputs, torch.Tensor):
            adapted = adapter(outputs)
            return adapted
        elif isinstance(outputs, tuple):
            adapted = adapter(outputs[0])
            return (adapted,) + outputs[1:]
        else:
            return outputs
    layer.forward = types.MethodType(forward_with_adapter, layer)

def wrap_decoder_layer(layer, adapter):
    original_forward = layer.forward
    def forward_with_adapter(self, hidden_states, *args, **kwargs):
        outputs = original_forward(hidden_states, *args, **kwargs)
        if isinstance(outputs, tuple):
            hs = outputs[0]
            adapted = adapter(hs)
            return (adapted,) + outputs[1:]
        elif isinstance(outputs, torch.Tensor):
            adapted = adapter(outputs)
            return adapted
        else:
            return outputs
    layer.forward = types.MethodType(forward_with_adapter, layer)

In [ ]:
# ============================================================
# BART with In-Layer Adapters
# ============================================================
class BARTWithAdapters(nn.Module):
    def __init__(self, backbone=BACKBONE, adapter_type="residual",
                 n_adapter_layers=3, adapter_bottleneck=64, device=DEVICE):
        super().__init__()
        self.device = device
        print("Loading BART backbone:", backbone)
        # TODO: Load pre-trained BART backbone and tokenizer
        # TODO: Freeze all model parameters except output embeddings
        # TODO: Get encoder and decoder layers
        # TODO: Identify target layer indices for adapter insertion
        # TODO: Initialize encoder and decoder adapter modules (Residual/Sparse)
        # TODO: Wrap chosen layers with corresponding adapters
        # TODO: Set adapter parameters to require gradient updates
        # TODO: Move entire model to specified device

    def forward(self, input_ids, attention_mask,
                decoder_input_ids=None, decoder_attention_mask=None, labels=None):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            labels=labels,
            use_cache=False,
            return_dict=True
        )
       # TODO: Return dictionary with loss and logits

    def generate_summary(self, text: str, max_length=120, num_beams=4):
        # TODO: Tokenize input text
        # TODO: Generate summary using model.generate()
        # TODO: Decode generated token IDs into text

In [ ]:
# ============================================================
# Instantiate model
# ============================================================
model = BARTWithAdapters(backbone=BACKBONE, adapter_type=ADAPTER_TYPE,
                         n_adapter_layers=N_ADAPTER_LAYERS,
                         adapter_bottleneck=ADAPTER_BOTTLENECK, device=DEVICE)

# ============================================================
# Dataset — MS² (allenai/mslr2022, config ms2)
# ============================================================
print("Loading MS² dataset (allenai/mslr2022, config ms2)...")
ds = load_dataset("allenai/mslr2022", "ms2")
train_ds = ds["train"]
val_ds = ds["validation"]
print("Sizes:", len(train_ds), len(val_ds))

if TRAIN_SUBSET:
    train_ds = train_ds.select(range(min(TRAIN_SUBSET, len(train_ds))))
if VAL_SUBSET:
    val_ds = val_ds.select(range(min(VAL_SUBSET, len(val_ds))))

def hf_row_to_example(row):
    bg = row.get("background", "") or ""
    abstract_list = row.get("abstract", []) or []
    tgt = row.get("target", "") or row.get("summary", "") or ""
    return {"background": bg, "abstracts": abstract_list, "target": tgt}

train_examples = [hf_row_to_example(r) for r in train_ds]
val_examples = [hf_row_to_example(r) for r in val_ds]
print("Prepared: train", len(train_examples), "val", len(val_examples))

In [ ]:
# ============================================================
# Collate
# ============================================================
def collate_fn(batch):
    inputs, targets = [], []
    for ex in batch:
        txt = flatten_abstracts_to_text(ex["background"], ex["abstracts"])
        inputs.append(txt)
        targets.append(ex["target"])
    tok_inputs = tokenize_texts(model.tokenizer, inputs, MAX_INPUT_LENGTH)
    tok_targets = tokenize_texts(model.tokenizer, targets, MAX_TARGET_LENGTH)

    pad_id = model.tokenizer.pad_token_id
    labels = tok_targets["input_ids"].clone()
    labels[labels == pad_id] = -100
    labels = labels.long()
    decoder_input_ids = model.model.prepare_decoder_input_ids_from_labels(labels)
    decoder_attention_mask = (decoder_input_ids != pad_id).long()

    return {
        "input_ids": tok_inputs["input_ids"].to(DEVICE),
        "attention_mask": tok_inputs["attention_mask"].to(DEVICE),
        "decoder_input_ids": decoder_input_ids.to(DEVICE),
        "decoder_attention_mask": decoder_attention_mask.to(DEVICE),
        "labels": labels.to(DEVICE),
    }

train_loader = DataLoader(train_examples, batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_examples, batch_size=1,
                        shuffle=False, collate_fn=collate_fn)

In [ ]:
# ============================================================
# Optimizer
# ============================================================
opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

# ============================================================
# Training loop
# ============================================================
print("Starting training ...")
for epoch in range(EPOCHS):
    model.train()
    cum_loss, steps = 0.0, 0
    for batch in tqdm(train_loader, desc=f"Train ep{epoch}"):
        # TODO: Zero optimizer gradients
        # TODO: Forward pass through model to get loss

        # TODO: If adapter type is "sparse", compute L1 regularization penalty
        #       - Iterate through model modules
        #       - Accumulate L1 loss for all SparseAdapters
        #       - Add weighted penalty to main loss

        # TODO: Backpropagate gradients
        # TODO: Clip gradients to prevent exploding values
        # TODO: Perform optimizer step to update trainable parameters
        if steps % 20 == 0:
            print(f"Epoch {epoch} step {steps} loss {loss.item():.4f}")
    print(f"Epoch {epoch} avg loss {(cum_loss/steps) if steps>0 else 0.0:.4f}")
    torch.save(model.state_dict(),
               os.path.join(CHECKPOINT_DIR, f"bart_adapters_{ADAPTER_TYPE}_epoch{epoch}.pt"))

    # Quick validation (small subset)
    # TODO: Set model to evaluation mode
    # TODO: Select a small subset of validation examples
    # TODO: Disable gradient computation for evaluation
    # TODO: For each validation sample:
    #           - Flatten abstracts into text input
    #           - Generate summary using the model
    #           - Compute ROUGE metrics for evaluation
    # TODO: Compute and print average ROUGE-1, ROUGE-2, and ROUGE-L score

In [ ]:
# ============================================================
# Final Evaluation
# ============================================================
print("Running final evaluation ...")
model.eval()
# TODO: Model evaluation on validation set

torch.save(model.state_dict(),
           os.path.join(CHECKPOINT_DIR, f"bart_adapters_{ADAPTER_TYPE}_final.pt"))
print("Done......................................................")

## PART 4

In [ ]:
# ============================================================
# Full Fine-tuning: facebook/bart-base on MS² (allenai/mslr2022)
# ============================================================

# ============================================================
# Model and Tokenizer
# ============================================================
print("Loading BART model and tokenizer...")
model = BartForConditionalGeneration.from_pretrained(BACKBONE).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(BACKBONE, use_fast=True)

# ============================================================
# Dataset
# ============================================================
print("Loading MS² dataset (allenai/mslr2022, config ms2)...")
ds = load_dataset("allenai/mslr2022", "ms2")
train_ds, val_ds = ds["train"], ds["validation"]

if TRAIN_SUBSET:
    train_ds = train_ds.select(range(min(TRAIN_SUBSET, len(train_ds))))
if VAL_SUBSET:
    val_ds = val_ds.select(range(min(VAL_SUBSET, len(val_ds))))

def to_examples(split):
    out = []
    for r in split:
        out.append({
            "background": r.get("background", "") or "",
            "abstracts": r.get("abstract", []) or [],
            "target": r.get("target", "") or r.get("summary", "") or ""
        })
    return out

train_examples = to_examples(train_ds)
val_examples = to_examples(val_ds)
print("Prepared:", len(train_examples), "train,", len(val_examples), "val")

# ============================================================
# Collate Function
# ============================================================
def collate_fn(batch):
    inputs, targets = [], []
    for ex in batch:
        txt = flatten_abstracts_to_text(ex["background"], ex["abstracts"])
        inputs.append(txt)
        targets.append(ex["target"])
    tok_inputs = tokenize_texts(tokenizer, inputs, MAX_INPUT_LENGTH)
    tok_targets = tokenize_texts(tokenizer, targets, MAX_TARGET_LENGTH)
    pad_id = tokenizer.pad_token_id
    labels = tok_targets["input_ids"].clone()
    labels[labels == pad_id] = -100
    labels = labels.long()
    dec_in = model.prepare_decoder_input_ids_from_labels(labels)
    dec_mask = (dec_in != pad_id).long()
    return {
        "input_ids": tok_inputs["input_ids"].to(DEVICE),
        "attention_mask": tok_inputs["attention_mask"].to(DEVICE),
        "decoder_input_ids": dec_in.to(DEVICE),
        "decoder_attention_mask": dec_mask.to(DEVICE),
        "labels": labels.to(DEVICE)
    }

train_loader = DataLoader(train_examples, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_examples, batch_size=1, shuffle=False, collate_fn=collate_fn)

# ============================================================
# Optimizer
# ============================================================
opt = torch.optim.AdamW(model.parameters(), lr=LR)

# ============================================================
# Training Loop
# ============================================================
print("Starting full fine-tuning...")
for epoch in range(EPOCHS):
    # TODO: complete training loop

    # quick validation check
    model.eval()
    with torch.no_grad():
        # TODO: complete quick validation

# ============================================================
# Final Evaluation
# ============================================================
print("\nRunning final evaluation...")
model.eval()
all_scores = []
# TODO: complete evaluation

torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "bart_full_final.pt"))
print("Full fine-tuning complete.")

In [ ]:
#------------------------------------------------------------------------------
#------------------------------------------------------------------------------
#------------------------------------------------------------------------------

## Question 2: XPROMPT-inspired pruning for efficient soft prompt tuning.

In [ ]:
!pip -q install accelerate --upgrade

## XPROMPT on SuperGLUE COPA - One classification task implementation



In [ ]:
# One partial working code for 1 task, complete it and do remaining 4 task.

In [ ]:
# ---- BART-base version ----

import math, random, copy, numpy as np, torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    BartForConditionalGeneration,
    BartTokenizerFast,
    Adafactor
)
from tqdm import tqdm
from matplotlib.colors import ListedColormap

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# ------------------------
# Reproducibility
# ------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

In [ ]:
# ------------------------
# Dataset (COPA Meta Template)
# ------------------------
def build_copa(tokenizer, max_len=256):
    ds = load_dataset("super_glue", "copa")

    def preprocess(ex):
        q = "cause" if ex["question"] == "cause" else "effect"
        source = (
            f"choice1: {ex['choice1']}\n"
            f"choice2: {ex['choice2']}\n"
            f"premise: {ex['premise']}\n"
            f"question: {q}\n"
            f"answer:"
        )
        target = "choice1" if ex["label"] == 0 else "choice2"
        return {"source": source, "target": target}

    train = ds["train"].map(preprocess, remove_columns=ds["train"].column_names)
    val = ds["validation"].map(preprocess, remove_columns=ds["validation"].column_names)
    train.set_format(type=None, columns=["source", "target"])
    val.set_format(type=None, columns=["source", "target"])

    def collate(batch):
        src = tokenizer([b["source"] for b in batch],
                        padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        tgt = tokenizer([b["target"] for b in batch],
                        padding=True, truncation=True,
                        max_length=8, return_tensors="pt")
        labels = tgt["input_ids"]
        labels[labels == tokenizer.pad_token_id] = -100
        return {
            "input_ids": src["input_ids"],
            "attention_mask": src["attention_mask"],
            "labels": labels,
            "targets": [b["target"] for b in batch],
        }

    return train, val, collate

tokenizer = BartTokenizerFast.from_pretrained("facebook/bart-base")
train_ds, val_ds, collate_fn = build_copa(tokenizer)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, collate_fn=collate_fn)
print("Dataset ready with 'choice1'/'choice2' labels.")

In [ ]:
# ------------------------
# Soft-Prompt Wrapper (BART)
# ------------------------
class SoftPromptBart(nn.Module):
    def __init__(self, model, prompt_len=20, piece_splits=16, tokenizer=None):
        super().__init__()
        self.model = model
        for p in self.model.parameters():
            p.requires_grad = False

        d_model = self.model.config.d_model
        self.prompt_len = prompt_len
        self.piece_splits = piece_splits
        assert d_model % piece_splits == 0, "d_model must divide evenly into piece_splits"

        # BART shares embeddings (model.model.shared); encoder also has embed_tokens
        vocab_ids = torch.randint(0, tokenizer.vocab_size, (prompt_len,))
        with torch.no_grad():
            init_emb = self.model.model.shared(vocab_ids.to(self.model.device))
        self.soft_prompts = nn.Parameter(init_emb.clone())

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        # Use shared embedding for input_ids
        token_emb = self.model.model.shared(input_ids)
        B = token_emb.size(0)
        prompts = self.soft_prompts.unsqueeze(0).expand(B, -1, -1)
        emb = torch.cat([prompts, token_emb], dim=1)

        if attention_mask is None:
            attention_mask = torch.ones(input_ids.size(), device=input_ids.device)
        prompt_mask = torch.ones(B, self.prompt_len, device=attention_mask.device, dtype=attention_mask.dtype)
        enc_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        return self.model(inputs_embeds=emb, attention_mask=enc_mask, labels=labels)

    @torch.no_grad()
    def generate(self, input_ids=None, attention_mask=None, **gen_kwargs):
        token_emb = self.model.model.shared(input_ids)
        B = token_emb.size(0)
        prompts = self.soft_prompts.unsqueeze(0).expand(B, -1, -1)
        emb = torch.cat([prompts, token_emb], dim=1)

        if attention_mask is None:
            attention_mask = torch.ones(input_ids.size(), device=input_ids.device)
        prompt_mask = torch.ones(B, self.prompt_len, device=attention_mask.device, dtype=attention_mask.dtype)
        enc_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        # BART config provides decoder_start_token_id; no change needed
        return self.model.generate(inputs_embeds=emb, attention_mask=enc_mask, **gen_kwargs)

    def token_importance(self):
        g = self.soft_prompts.grad.detach()
        return g.abs().mean(dim=1)

    def piece_importance(self):
        g = self.soft_prompts.grad.detach()
        P, d = g.shape
        g = g.view(P, self.piece_splits, d // self.piece_splits)
        return g.abs().mean(dim=2)

In [ ]:
# ------------------------
# Initialize model
# ------------------------
base = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to(device)
xp = SoftPromptBart(base, prompt_len=20, piece_splits=16, tokenizer=tokenizer).to(device)

# ------------------------
# Training Utilities
# ------------------------
def make_optim(params, lr=0.3):
    return Adafactor(params, lr=lr, scale_parameter=False,
                     relative_step=False, weight_decay=1e-5)

def train_epoch(model, loader, opt):
    model.train(); total = 0
    for batch in tqdm(loader, desc="Training", leave=False, disable=True):
        batch = {k: v.to(device) for k, v in batch.items()
                 if k in ["input_ids", "attention_mask", "labels"]}
        opt.zero_grad(set_to_none=True)
        out = model(**batch)
        loss = out.loss
        loss.backward()
        opt.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def evaluate_generation(model, tokenizer, dataset, batch_size=16, max_gen_len=8):
    model.eval(); correct, total = 0, 0
    for i in range(0, len(dataset), batch_size):
        chunk = dataset[i: i + batch_size]
        src = tokenizer(chunk["source"], padding=True, truncation=True,
                        max_length=256, return_tensors="pt").to(device)
        gen = model.generate(**src, max_length=max_gen_len)
        preds = tokenizer.batch_decode(gen, skip_special_tokens=True)
        labels = [l.strip().lower() for l in chunk["target"]]
        preds = [p.strip().lower() for p in preds]
        correct += sum(p == l for p, l in zip(preds, labels))
        total += len(labels)
    return correct / max(1, total)

# ------------------------
# Stage 1: Prompt Tuning
# ------------------------
epochs = 90
opt = make_optim([xp.soft_prompts], lr=0.3)

print("\n--- Stage 1: Prompt Tuning ---")
for ep in range(epochs):
    loss = train_epoch(xp, train_loader, opt)
    if (ep+1) % 10 == 0:
        acc = evaluate_generation(xp, tokenizer, val_ds)
        print(f"Epoch {ep+1}/{epochs} | loss={loss:.4f} | dev_acc={acc*100:.2f}%")

pre_acc = evaluate_generation(xp, tokenizer, val_ds)
print(f"\nDev accuracy BEFORE pruning: {pre_acc*100:.2f}%")
initial_prompt = xp.soft_prompts.detach().clone()

# ------------------------
# Gradient-based Importance + Pruning
# ------------------------
def accumulate_importance_grads(model, loader):
    model.zero_grad(set_to_none=True)
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()
                 if k in ["input_ids", "attention_mask", "labels"]}
        out = model(**batch)
        (out.loss / len(loader)).backward()
    tok_imp = model.token_importance()
    piece_imp = model.piece_importance()
    return tok_imp, piece_imp

def build_masks(tok_imp, piece_imp, prune_token, prune_piece):
    P, K = tok_imp.numel(), piece_imp.size(1)
    tok_keep = torch.zeros(P, dtype=torch.bool, device=tok_imp.device)
    keep_t = max(1, int((1 - prune_token) * P))
    _, top_t = torch.topk(tok_imp, k=keep_t)
    tok_keep[top_t] = True
    piece_keep = torch.zeros_like(piece_imp, dtype=torch.bool)
    keep_p = max(1, int((1 - prune_piece) * K))
    for i in range(P):
        _, idx = torch.topk(piece_imp[i], k=keep_p)
        piece_keep[i, idx] = True
    return tok_keep, piece_keep

def apply_prune(model, tok_keep, piece_keep):
    with torch.no_grad():
        P, d = model.soft_prompts.shape
        K = piece_keep.size(1)
        chunk = d // K
        piece_mask_full = torch.repeat_interleave(piece_keep.float(), chunk, dim=1)
        tok_mask_full = tok_keep.unsqueeze(1).float().expand(P, d)
        full_mask = piece_mask_full * tok_mask_full
        model.soft_prompts.data *= full_mask
        kept = full_mask.sum().item()
        print(f"[Prune] kept {kept/(P*d)*100:.2f}% of prompt weights")
    return full_mask

In [ ]:

# ------------------------
# Stage 2 + 3: Hierarchical Pruning + Rewind + Retrain
# ------------------------
ratios = [0.1, 0.3]
best_acc, best_combo = -1.0, None
best_tok_imp = best_piece_imp = best_tok_keep = best_piece_keep = None

print("\n--- Stage 2 + 3: Pruning Sweep ---")
for rt in ratios:
    for rp in ratios:
        print(f"\nTrying token_prune={int(rt*100)}%, piece_prune={int(rp*100)}%")
        with torch.no_grad():
            xp.soft_prompts.copy_(initial_prompt)
        tok_imp, piece_imp = accumulate_importance_grads(xp, train_loader)
        tok_keep, piece_keep = build_masks(tok_imp, piece_imp, rt, rp)
        mask = apply_prune(xp, tok_keep, piece_keep)
        opt = make_optim([xp.soft_prompts], lr=0.3)
        for ep in tqdm(range(10), desc="Retraining", leave=False, disable=True):
            loss = train_epoch(xp, train_loader, opt)
        acc = evaluate_generation(xp, tokenizer, val_ds)
        print(f"  → Dev accuracy after retrain: {acc*100:.2f}%")
        if acc > best_acc:
            best_acc, best_combo = acc, (rt, rp)
            best_prompt = xp.soft_prompts.detach().clone()
            best_tok_imp, best_piece_imp = tok_imp.detach().clone(), piece_imp.detach().clone()
            best_tok_keep, best_piece_keep = tok_keep.detach().clone(), piece_keep.detach().clone()

if best_combo:
    print(f"\nBest pruning ratio = token {int(best_combo[0]*100)}%, piece {int(best_combo[1]*100)}% | acc={best_acc*100:.2f}%")
    with torch.no_grad():
        xp.soft_prompts.copy_(best_prompt)

print(f"\nFinal dev accuracy AFTER XPROMPT pruning: {best_acc*100:.2f}%")
print(f"Before pruning: {pre_acc*100:.2f}% | After pruning: {best_acc*100:.2f}%")

In [ ]:
# ------------------------
# Visualization
# ------------------------

# --- Token importance ---
imp_tok = best_tok_imp.cpu().numpy() * 1e6
plt.figure(figsize=(9, 3))
plt.bar(np.arange(len(imp_tok)), imp_tok, color='steelblue', edgecolor='black')
plt.title("Soft Prompt Token Importance (Before Pruning)")
plt.xlabel("Prompt Token Index")
plt.ylabel("Mean Grad Magnitude (×1e6)")
plt.tight_layout(); plt.show()

# --- Token retention ---
keep_tok = (best_tok_keep.cpu().numpy() > 0.5).astype(int)
colors = ['green' if k else 'red' for k in keep_tok]
plt.figure(figsize=(9, 1.4))
plt.bar(np.arange(len(keep_tok)), np.ones(len(keep_tok)), color=colors, edgecolor='black')
plt.title(f"Pruned (red) vs Kept (green) Tokens — Token {int(best_combo[0]*100)}%")
plt.xlabel("Prompt Token Index"); plt.yticks([]); plt.ylim(0, 1)
plt.tight_layout(); plt.show()

# --- Piece importance heatmap ---
imp_piece = best_piece_imp.cpu().numpy()
imp_piece = imp_piece / (imp_piece.max() + 1e-12)
plt.figure(figsize=(10, 3.8))
plt.imshow(imp_piece, cmap='Blues', aspect='auto')
plt.colorbar(label="Normalized Importance")
plt.title("Piece-level Importance per Token (Before Pruning)")
plt.xlabel("Piece Index (per-token split)")
plt.ylabel("Prompt Token Index")
plt.tight_layout(); plt.show()

# --- Piece retention heatmap ---
keep_piece = best_piece_keep.cpu().numpy().astype(int)
cmap = ListedColormap(['#cc1f1f', '#2aa22a'])
plt.figure(figsize=(10, 3.8))
plt.imshow(keep_piece, cmap=cmap, aspect='auto', vmin=0, vmax=1)
plt.colorbar(ticks=[0, 1], label="Retention (Red=Pruned, Green=Kept)")
plt.title(f"Piece Retention Map (After Pruning — Token {int(best_combo[0]*100)}%, Piece {int(best_combo[1]*100)}%)")
plt.xlabel("Piece Index"); plt.ylabel("Prompt Token Index")
plt.tight_layout(); plt.show()

# --- Distribution histogram (like XPROMPT Figure 7) ---
flat_scores = best_piece_imp.cpu().numpy().ravel()
flat_scores = (flat_scores - flat_scores.min()) / (np.ptp(flat_scores) + 1e-12)
bins = np.linspace(0, 1, 11)
counts, edges = np.histogram(flat_scores, bins=bins)
plt.figure(figsize=(8, 4.5))
bars = plt.bar(edges[:-1], counts, width=np.diff(edges), align='edge',
               color='#a8e067', edgecolor='black')
for b, c in zip(bars, counts):
    if c > 0:
        plt.text(b.get_x() + b.get_width()/2, c + 0.5, str(int(c)),
                 ha='center', va='bottom', fontsize=9)
labels = [f"{edges[i]:.1f}-{edges[i+1]:.1f}" for i in range(len(edges)-1)]
plt.xticks(edges[:-1], labels)
plt.xlabel("Importance score"); plt.ylabel("Frequency")
plt.title("Distribution of Prompt Token-Piece Importance Scores")
plt.legend(["PromptTokenPiece"])
plt.tight_layout(); plt.show()

## XPROMPT on SuperGLUE WiC - One classification task implementation

In [ ]:
# TODO: complete code WiC following code above for SuperGLUE COPA

# XPROMPT on SuperGLUE WSC - One classification task implementation

In [ ]:
# TODO: complete code WSC following code above for SuperGLUE COPA

## XPROMPT on SuperGLUE CB - One classification task implementation



In [ ]:
# TODO: complete code CB following code above for SuperGLUE COPA

## XPROMPT on SuperGLUE  RTE - One classification task implementation



In [ ]:
# TODO: complete code RTE following code above for SuperGLUE COPA

In [ ]:
# Conclusion

# write code here